In [1]:
# MULTI-DISEASE PREDICTION MODEL - XGBOOST (UPDATED)
# Changes: 1) Removed Family_History, 2) Individual symptom features instead of count

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("MULTI-DISEASE PREDICTION MODEL - XGBOOST")
print("="*70)

# Load dataset
df = pd.read_csv('patient_ds2.csv')
print(f"\nDataset loaded: {df.shape[0]} patients, {df.shape[1]} features")

# Define Group 2+3 diseases
group23_diseases = [
    'COVID-19', 'Influenza', 'Typhoid', 'Dengue', 'Malaria', 'Hepatitis',
    'Asthma', 'Bronchitis', 'Pneumonia', 'Tuberculosis'
]

print(f"\nTraining for {len(group23_diseases)} diseases:")
for i, disease in enumerate(group23_diseases, 1):
    print(f"  {i}. {disease}")

# Filter dataset
df_filtered = df[df['Diagnosis'].isin(group23_diseases)].copy()
print(f"\nFiltered dataset: {df_filtered.shape[0]} patients")

# PREPROCESSING
print("\n" + "-"*70)
print("PREPROCESSING DATA")
print("-"*70)

df_model = df_filtered.drop('Patient_ID', axis=1).copy()

# Handle missing values
for column in df_model.columns:
    if column == 'Diagnosis':
        continue
    if df_model[column].dtype == 'object':
        mode_value = df_model[column].mode()[0] if not df_model[column].mode().empty else 'Unknown'
        df_model[column].fillna(mode_value, inplace=True)
    else:
        median_value = df_model[column].median()
        df_model[column].fillna(median_value, inplace=True)

# Extract Blood Pressure
def extract_bp(bp_str):
    if pd.isna(bp_str) or bp_str == '':
        return 120, 80
    try:
        systolic, diastolic = str(bp_str).split('/')
        return float(systolic), float(diastolic)
    except:
        return 120, 80

bp_data = df_model['Blood_Pressure'].apply(extract_bp)
df_model['Systolic_BP'] = [bp[0] for bp in bp_data]
df_model['Diastolic_BP'] = [bp[1] for bp in bp_data]
df_model.drop('Blood_Pressure', axis=1, inplace=True)

# Create Age Group
df_model['Age_Group'] = pd.cut(df_model['Age'],
                               bins=[0, 40, 55, 70, 100],
                               labels=['Young', 'Middle', 'Senior', 'Elderly'])

# INDIVIDUAL SYMPTOM FEATURES (instead of Symptom_Count)
# Define common symptoms for these diseases
all_symptoms = [
    'Fever', 'Cough', 'Shortness of Breath', 'Fatigue', 'Headache',
    'Body Aches', 'Sore Throat', 'Nausea', 'Vomiting', 'Diarrhea',
    'Loss of Taste', 'Loss of Smell', 'Chest Pain', 'Chills', 'Rash'
]

# Extract individual symptoms from Symptoms column
for symptom in all_symptoms:
    df_model[f'Has_{symptom.replace(" ", "_")}'] = df_model['Symptoms'].apply(
        lambda x: 1 if pd.notna(x) and symptom.lower() in str(x).lower() else 0
    )

df_model.drop('Symptoms', axis=1, inplace=True)

# Remove Family_History
if 'Family_History' in df_model.columns:
    df_model.drop('Family_History', axis=1, inplace=True)
    print("Removed: Family_History")

# Encode categorical variables
label_encoders = {}
categorical_columns = ['Gender', 'Smoking_Status', 'Alcohol_Use',
                      'Physical_Activity_Level', 'Age_Group']

for column in categorical_columns:
    if column in df_model.columns:
        le = LabelEncoder()
        df_model[column] = le.fit_transform(df_model[column].astype(str))
        label_encoders[column] = le

# Encode target
disease_encoder = LabelEncoder()
y = disease_encoder.fit_transform(df_model['Diagnosis'])
X = df_model.drop('Diagnosis', axis=1)

print(f"\nPreprocessing complete!")
print(f"Features: {X.shape[1]}")
print(f"Symptom features: {len(all_symptoms)}")
print(f"Target classes: {len(disease_encoder.classes_)}")

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nData split:")
print(f"  Training: {X_train.shape[0]} samples")
print(f"  Testing: {X_test.shape[0]} samples")

# TRAIN XGBOOST MODEL
print("\n" + "-"*70)
print("TRAINING XGBOOST MODEL")
print("-"*70)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='multi:softmax',
    num_class=len(disease_encoder.classes_),
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"\nXGBoost Training Complete!")
print(f"Accuracy: {accuracy_xgb:.3f} ({accuracy_xgb*100:.1f}%)")

# EVALUATION
print("\n" + "-"*70)
print("CLASSIFICATION REPORT")
print("-"*70)
print(classification_report(y_test, y_pred_xgb,
                          target_names=disease_encoder.classes_,
                          zero_division=0))

# Feature Importance
print("\nTOP 15 IMPORTANT FEATURES:")
print("-"*70)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

for idx, row in feature_importance.iterrows():
    print(f"{row['feature']:<35} {row['importance']:.4f}")

# SAVE MODEL AND ARTIFACTS
print("\n" + "-"*70)
print("SAVING MODEL TO PKL FILE")
print("-"*70)

model_artifacts = {
    'model': xgb_model,
    'disease_encoder': disease_encoder,
    'label_encoders': label_encoders,
    'categorical_columns': categorical_columns,
    'feature_columns': list(X.columns),
    'diseases': list(disease_encoder.classes_),
    'accuracy': accuracy_xgb,
    'group23_diseases': group23_diseases,
    'symptom_list': all_symptoms
}

# Save to pickle file
with open('MDPv2.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Model saved: MDPv2.pkl")
print(f"Model accuracy: {accuracy_xgb*100:.1f}%")
print(f"Diseases: {list(disease_encoder.classes_)}")
print(f"Number of features: {len(X.columns)}")
print(f"Symptom features: {len(all_symptoms)}")

print("\n" + "="*70)
print("MODEL TRAINING COMPLETE")
print("="*70)


MULTI-DISEASE PREDICTION MODEL - XGBOOST

Dataset loaded: 10345 patients, 14 features

Training for 10 diseases:
  1. COVID-19
  2. Influenza
  3. Typhoid
  4. Dengue
  5. Malaria
  6. Hepatitis
  7. Asthma
  8. Bronchitis
  9. Pneumonia
  10. Tuberculosis

Filtered dataset: 4475 patients

----------------------------------------------------------------------
PREPROCESSING DATA
----------------------------------------------------------------------
Removed: Family_History

Preprocessing complete!
Features: 27
Symptom features: 15
Target classes: 10

Data split:
  Training: 3580 samples
  Testing: 895 samples

----------------------------------------------------------------------
TRAINING XGBOOST MODEL
----------------------------------------------------------------------

XGBoost Training Complete!
Accuracy: 0.913 (91.3%)

----------------------------------------------------------------------
CLASSIFICATION REPORT
----------------------------------------------------------------------
  

In [2]:
# LOAD AND TEST UPDATED MODEL

import pickle
import pandas as pd
import numpy as np

# Load the saved model
with open('MDPv2.pkl', 'rb') as f:
    artifacts = pickle.load(f)

model = artifacts['model']
disease_encoder = artifacts['disease_encoder']
label_encoders = artifacts['label_encoders']
categorical_columns = artifacts['categorical_columns']
feature_columns = artifacts['feature_columns']
symptom_list = artifacts['symptom_list']

print("Model loaded successfully!")
print(f"Accuracy: {artifacts['accuracy']*100:.1f}%")
print(f"Available symptoms: {symptom_list}")

# Test prediction function for frontend
def predict_disease(age, gender, heart_rate, body_temperature, respiratory_rate,
                   oxygen_saturation, systolic_bp, diastolic_bp,
                   smoking_status, alcohol_use, physical_activity_level,
                   checked_symptoms):
    """
    Predict disease from patient data

    Parameters:
    checked_symptoms: list of symptoms checked by user in frontend
                     e.g., ['Fever', 'Cough', 'Shortness of Breath']
    """

    # Create patient data
    patient_data = {
        'Age': age,
        'Gender': gender,
        'Heart_Rate': heart_rate,
        'Body_Temperature': body_temperature,
        'Respiratory_Rate': respiratory_rate,
        'Oxygen_Saturation': oxygen_saturation,
        'Systolic_BP': systolic_bp,
        'Diastolic_BP': diastolic_bp,
        'Smoking_Status': smoking_status,
        'Alcohol_Use': alcohol_use,
        'Physical_Activity_Level': physical_activity_level
    }

    # Add Age Group
    if age < 40:
        patient_data['Age_Group'] = 'Young'
    elif age < 55:
        patient_data['Age_Group'] = 'Middle'
    elif age < 70:
        patient_data['Age_Group'] = 'Senior'
    else:
        patient_data['Age_Group'] = 'Elderly'

    # Add individual symptom features based on checkboxes
    for symptom in symptom_list:
        symptom_key = f'Has_{symptom.replace(" ", "_")}'
        patient_data[symptom_key] = 1 if symptom in checked_symptoms else 0

    patient_df = pd.DataFrame([patient_data])

    # Encode categorical variables
    for col in categorical_columns:
        if col in patient_df.columns and col in label_encoders:
            try:
                patient_df[col] = label_encoders[col].transform(patient_df[col])
            except:
                patient_df[col] = 0

    # Ensure correct column order
    patient_df = patient_df[feature_columns]

    # Predict
    prediction = model.predict(patient_df)[0]
    probabilities = model.predict_proba(patient_df)[0]

    predicted_disease = disease_encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]

    all_probs = {disease_encoder.classes_[i]: probabilities[i]
                 for i in range(len(disease_encoder.classes_))}

    return predicted_disease, confidence, all_probs

# TEST EXAMPLE
print("\n" + "-"*70)
print("TEST PREDICTION")
print("-"*70)

# Simulate frontend checkbox selection
checked_symptoms = ['Fever', 'Cough', 'Shortness of Breath', 'Fatigue', 'Chest Pain']

disease, conf, all_probs = predict_disease(
    age=58,
    gender='Male',
    heart_rate=115,
    body_temperature=39.2,
    respiratory_rate=28,
    oxygen_saturation=88,
    systolic_bp=110,
    diastolic_bp=72,
    smoking_status='Former',
    alcohol_use='No',
    physical_activity_level='Low',
    checked_symptoms=checked_symptoms
)

print(f"\nPatient symptoms checked: {', '.join(checked_symptoms)}")
print(f"\nPredicted Disease: {disease}")
print(f"Confidence: {conf:.2%}")
print("\nTop 3 probabilities:")
sorted_probs = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)[:3]
for d, p in sorted_probs:
    print(f"  {d}: {p:.2%}")


Model loaded successfully!
Accuracy: 91.3%
Available symptoms: ['Fever', 'Cough', 'Shortness of Breath', 'Fatigue', 'Headache', 'Body Aches', 'Sore Throat', 'Nausea', 'Vomiting', 'Diarrhea', 'Loss of Taste', 'Loss of Smell', 'Chest Pain', 'Chills', 'Rash']

----------------------------------------------------------------------
TEST PREDICTION
----------------------------------------------------------------------

Patient symptoms checked: Fever, Cough, Shortness of Breath, Fatigue, Chest Pain

Predicted Disease: Pneumonia
Confidence: 99.85%

Top 3 probabilities:
  Pneumonia: 99.85%
  COVID-19: 0.14%
  Malaria: 0.00%
